In [2]:
# ==============================================================================
# STEP 0: INSTALL REQUIRED LIBRARIES
# ==============================================================================
!pip install ultralytics albumentations

# Import necessary libraries
import os
import zipfile
import cv2
from pathlib import Path
from ultralytics import YOLO
from google.colab import files
from google.colab.patches import cv2_imshow
import yaml
import glob
from collections import Counter
import albumentations as A

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# ==============================================================================
# STEP 1: GET THE DATASET (USE EXISTING OR UPLOAD NEW)
# ==============================================================================
ZIP_FILE_NAME = "P-ID Symbols.v1i.yolov8.zip"  # Your ZIP file name
zip_path = Path(ZIP_FILE_NAME)

if not zip_path.is_file():
    print(f"'{zip_path}' not found. Please upload your zipped dataset.")
    uploaded = files.upload()
    if not uploaded or zip_path.name not in uploaded:
        raise Exception(f"Upload failed or file name mismatch.")
    print(f"Successfully uploaded '{zip_path.name}'.")
else:
    print(f"Found existing file '{zip_path.name}'. Skipping upload.")

print(f"Unzipping '{zip_path.name}'...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')
print("Unzip complete.")

# Find data.yaml recursively, assuming it’s in the unzip root folder
unzip_root = Path('./P-ID Symbols.v1i.yolov8')  # Adjust if folder name differs
data_yaml_path = next(unzip_root.rglob('data.yaml'), None)
if data_yaml_path is None:
    # Fallback to broader search if not found
    data_yaml_path = next(Path('.').rglob('data.yaml'), None)
    if data_yaml_path is None:
        raise FileNotFoundError("Could not find data.yaml in the unzipped folder. Check the ZIP structure or folder name.")
print(f"Found data.yaml at: {data_yaml_path}")

# Load and update data.yaml with absolute paths
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml['names']
dataset_root = data_yaml_path.parent
data_yaml['train'] = str(dataset_root / 'train/images')
data_yaml['val'] = str(dataset_root / 'valid/images')
data_yaml['test'] = str(dataset_root / 'test/images')

# Save updated data.yaml
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)
print(f"Updated data.yaml paths: train={data_yaml['train']}, val={data_yaml['val']}, test={data_yaml['test']}")

'P-ID Symbols.v1i.yolov8.zip' not found. Please upload your zipped dataset.


Saving P-ID Symbols.v1i.yolov8.zip to P-ID Symbols.v1i.yolov8.zip
Successfully uploaded 'P-ID Symbols.v1i.yolov8.zip'.
Unzipping 'P-ID Symbols.v1i.yolov8.zip'...
Unzip complete.
Found data.yaml at: data.yaml
Updated data.yaml paths: train=train/images, val=valid/images, test=test/images


In [4]:
# ==============================================================================
# STEP 0.5: CHECK CLASS DISTRIBUTION (TO IDENTIFY IMBALANCE)
# ==============================================================================
def get_class_distribution(label_dir):
    class_counts = Counter()
    label_files = glob.glob(os.path.join(label_dir, '**/*.txt'), recursive=True)  # Recursive search
    if not label_files:
        print(f"Warning: No label files found in {label_dir}. Check directory structure.")
    for label_file in label_files:
        with open(label_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                if line.strip():
                    class_id = int(line.split()[0])
                    class_counts[class_id] += 1
    return class_counts

# Get train label dir from updated data_yaml
train_label_dir = str(Path(data_yaml['train']).parent / 'labels')  # Adjust to 'labels' folder
print(f"Checking class distribution in: {train_label_dir}")
class_dist = get_class_distribution(train_label_dir)

print("Class Distribution Report (Training Set):")
for cls_id, count in sorted(class_dist.items()):
    name = class_names[cls_id] if cls_id < len(class_names) else f"Unknown ({cls_id})"
    print(f"- {name} (ID {cls_id}): {count} instances")
rare_classes = [name for cls_id, count in class_dist.items() if count < 5]
if rare_classes:
    print(f"\nRare Classes (<5 instances): {', '.join(rare_classes)} - Consider adding more examples.")

Checking class distribution in: train/labels
Class Distribution Report (Training Set):
- 3 Way Ball Valve (ID 0): 72 instances
- 3 Way Gate Valve (ID 1): 546 instances
- 3 Way Globe Valve (ID 2): 71 instances
- 4 Way Ball Valve (ID 3): 63 instances
- 4 Way Gate Valve (ID 4): 276 instances
- Alkyalation (ID 5): 9 instances
- Angle Blowdown (ID 6): 90 instances
- Angle GlobeValve (ID 7): 97 instances
- Angle Valve (ID 8): 417 instances
- Automatic Stoker (ID 9): 7 instances
- Axial Compressor (ID 10): 26 instances
- Axical COMP (ID 11): 9 instances
- BackPressure Regulator (ID 12): 243 instances
- Balance Diaphragm Gate Valve (ID 13): 108 instances
- Ball Valve (ID 14): 1571 instances
- Bleeder Valve (ID 15): 520 instances
- Boom Loader (ID 16): 9 instances
- Butterfly Valve (ID 17): 834 instances
- Cavity Pump (ID 18): 18 instances
- Centrifugal Blower (ID 19): 9 instances
- Centrifugal Compressor (ID 20): 60 instances
- Centrifugal Pump (ID 21): 1473 instances
- Check Valve (ID 22): 99

In [ ]:
# ==============================================================================
# STEP 2: TRAIN THE YOLOv8 MODEL (IMPROVED)
# ==============================================================================
model = YOLO('yolov8m.pt')  # Upgraded to medium for better capacity with 180 classes

# Custom augmentation pipeline (fixed Flip to HorizontalFlip and VerticalFlip)
transform = A.Compose([
    A.Rotate(limit=30, p=0.5),
    A.GaussianBlur(p=0.3),
    A.GaussNoise(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomCrop(width=640, height=640, p=0.2),
])

print("Starting model training...")
results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=30,
    augment=True,
    optimizer='AdamW',
    lr0=0.001,
    momentum=0.9,
    weight_decay=0.0005,
    copy_paste=0.1,
    mixup=0.1,
    name='pid_symbols_yolov8_improved'
)
print("Training finished.")

Starting model training...
Ultralytics 8.3.184 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=pid_symbols_yolov8_improved, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspective=0.0, plots=T

Overriding model.yaml nc=80 with nc=180

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralyti

 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3879916  ultralytics.nn.modules.head.Detect           [180, [192, 384, 576]]        
Model summary: 169 layers, 25,960,540 parameters, 25,960,524 gradients, 79.6 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 864.4±369.7 MB/s, size: 29.8 KB)


train: Scanning /content/train/labels... 2619 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2619/2619 [00:01<00:00, 2399.82it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     68/100      7.37G      1.078     0.6668      1.035        176        640: 100%|██████████| 164/164 [01:33<00:00,  1.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.06it/s]

                   all         95        729      0.716      0.625      0.666      0.366



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     69/100      7.04G      1.068     0.6598      1.041        169        640:  91%|█████████▏| 150/164 [01:25<00:07,  1.77it/s]

In [2]:
from google.colab import files
save_dir = 'runs/train/pid_symbols_yolov8_improved/weights'
for file in ['last.pt', 'best.pt']:
    if os.path.exists(f'{save_dir}/{file}'):
        files.download(f'{save_dir}/{file}')
files.download('data.yaml')
print("Files downloaded if present.")

FileNotFoundError: Cannot find file: data.yaml